WEBSITE DATA EXPORT NOTEBOOK
Run this notebook after all analysis is complete to export JSON files for the ArXiv Showcase website.

Prerequisites:
- All pkl files from your analysis pipeline
- Run cells in order

Output:
- All JSON files saved to 'results/' directory
- Copy these to your website's src/data/ folder

In [7]:
# imports and setup

import json
import os
import pickle
import numpy as np
import pandas as pd

# Create output directory
output_dir = 'results'
os.makedirs(output_dir, exist_ok=True)

def to_native(val):
    """Convert numpy types to Python native types for JSON serialization"""
    if isinstance(val, (np.integer, np.int64, np.int32)):
        return int(val)
    elif isinstance(val, (np.floating, np.float64, np.float32)):
        return float(val)
    elif isinstance(val, np.ndarray):
        return val.tolist()
    elif isinstance(val, dict):
        return {k: to_native(v) for k, v in val.items()}
    elif isinstance(val, list):
        return [to_native(v) for v in val]
    return val

def is_meaningful_term(term):
    """Check if term is meaningful (not LaTeX or too short)"""
    if len(term) <= 2:
        return False
    latex_prefixes = ['math', 'frac', 'text', 'emph', 'cite', 'ref', 'left', 'right', 'begin', 'end', 'infty', 'geq', 'leq']
    for prefix in latex_prefixes:
        if term.startswith(prefix):
            return False
    return True

# Category and domain name mappings
CATEGORY_NAMES = {
    'hep-ph': 'High Energy Physics - Phenomenology',
    'hep-th': 'High Energy Physics - Theory',
    'hep-ex': 'High Energy Physics - Experiment',
    'astro-ph': 'Astrophysics',
    'astro-ph.GA': 'Astrophysics - Galaxies',
    'astro-ph.CO': 'Astrophysics - Cosmology',
    'astro-ph.SR': 'Astrophysics - Solar/Stellar',
    'astro-ph.HE': 'Astrophysics - High Energy',
    'cond-mat': 'Condensed Matter',
    'cond-mat.mes-hall': 'Condensed Matter - Mesoscale',
    'cond-mat.mtrl-sci': 'Condensed Matter - Materials',
    'cond-mat.stat-mech': 'Condensed Matter - Stat Mech',
    'quant-ph': 'Quantum Physics',
    'gr-qc': 'General Relativity & Quantum Cosmology',
    'nucl-th': 'Nuclear Theory',
    'nucl-ex': 'Nuclear Experiment',
    'math-ph': 'Mathematical Physics',
    'nlin': 'Nonlinear Sciences',
    'math.AG': 'Math - Algebraic Geometry',
    'math.NT': 'Math - Number Theory',
    'math.CO': 'Math - Combinatorics',
    'math.AP': 'Math - Analysis/PDEs',
    'math.DG': 'Math - Differential Geometry',
    'math.PR': 'Math - Probability',
    'math.RT': 'Math - Representation Theory',
    'cs.LG': 'CS - Machine Learning',
    'cs.CV': 'CS - Computer Vision',
    'cs.CL': 'CS - NLP/Computation & Language',
    'cs.AI': 'CS - Artificial Intelligence',
    'cs.NE': 'CS - Neural & Evolutionary',
    'cs.CR': 'CS - Cryptography',
    'cs.IT': 'CS - Information Theory',
    'cs.DS': 'CS - Data Structures & Algorithms',
    'cs.RO': 'CS - Robotics',
    'stat.ML': 'Statistics - Machine Learning',
    'stat.TH': 'Statistics - Theory',
    'eess.SP': 'EESS - Signal Processing',
    'eess.IV': 'EESS - Image & Video',
    'q-bio': 'Quantitative Biology',
    'q-fin': 'Quantitative Finance',
}

DOMAIN_NAMES = {
    'cs': 'Computer Science',
    'math': 'Mathematics',
    'physics': 'Physics',
    'stat': 'Statistics',
    'hep-ph': 'High Energy Physics',
    'hep-th': 'Theoretical Physics',
    'hep-ex': 'HEP Experiment',
    'hep-lat': 'HEP Lattice',
    'astro-ph': 'Astrophysics',
    'cond-mat': 'Condensed Matter',
    'quant-ph': 'Quantum Physics',
    'gr-qc': 'General Relativity',
    'nucl-th': 'Nuclear Theory',
    'nucl-ex': 'Nuclear Experiment',
    'nlin': 'Nonlinear Sciences',
    'math-ph': 'Mathematical Physics',
    'eess': 'Electrical Engineering',
    'q-bio': 'Quantitative Biology',
    'q-fin': 'Quantitative Finance',
    'econ': 'Economics',
}

# Domain mapping helper — derives consolidated domain from ArXiv category code prefix.
# Replaces top_level_domain column which used raw prefixes (gr-qc, astro-ph, hep-ph)
# causing physics/math clusters to show incorrect primary domain.
def get_consolidated_domain(cat_code):
    """Map ArXiv category code to consolidated domain bucket."""
    if not isinstance(cat_code, str): return 'other'
    if cat_code.startswith('cs.'): return 'cs'
    if cat_code.startswith('math.') or cat_code == 'math-ph': return 'math'
    if any(cat_code.startswith(p) for p in [
        'astro-ph', 'hep-', 'gr-qc', 'nucl-', 'physics.', 'quant-ph', 'cond-mat'
    ]): return 'physics'
    if cat_code.startswith('stat.'): return 'stats'
    if cat_code.startswith('eess.'): return 'eess'
    if cat_code.startswith('q-bio'): return 'bio'
    if cat_code.startswith('econ.'): return 'econ'
    return 'other'



def get_category_name(code):
    return CATEGORY_NAMES.get(code, code)

def get_domain_name(code):
    return DOMAIN_NAMES.get(code, code)

print("✓ Setup complete")
print(f"  Output directory: {output_dir}/")


✓ Setup complete
  Output directory: results/


In [8]:
# load all pkl files

# --- Load the main dataframe (your original cleaned data) ---
df_original = pd.read_pickle('data/processed/arxiv_text_cleaned.pkl')  # Adjust name if different

# --- Load cluster labels ---
with open('data/processed/cluster_labels_500d.pkl', 'rb') as f:
    cluster_labels_500d = pickle.load(f)

with open('data/processed/cluster_labels_300d.pkl', 'rb') as f:
    cluster_labels_300d = pickle.load(f)

# --- Create df_500d and df_300d by adding cluster_id to original df ---
df_500d = df_original.copy()
df_500d['cluster_id'] = cluster_labels_500d['cluster_id']

df_300d = df_original.copy()
df_300d['cluster_id'] = cluster_labels_300d['cluster_id']

# --- Load cluster profiles ---
with open('data/processed/cluster_profiles_500d.pkl', 'rb') as f:
    profiles_500d = pickle.load(f)

with open('data/processed/cluster_profiles_300d.pkl', 'rb') as f:
    profiles_300d = pickle.load(f)

# --- Load evaluation metrics ---
with open('data/processed/k_evaluation_metrics_500d.pkl', 'rb') as f:
    eval_metrics_500d = pickle.load(f)

with open('data/processed/k_evaluation_metrics_300d.pkl', 'rb') as f:
    eval_metrics_300d = pickle.load(f)

print("✓ All pkl files loaded")
print(f"  df_500d: {len(df_500d):,} papers")
print(f"  df_300d: {len(df_300d):,} papers")
print(f"  Clusters (500d): {profiles_500d['n_clusters']}")
print(f"  Clusters (300d): {profiles_300d['n_clusters']}")


✓ All pkl files loaded
  df_500d: 2,384,617 papers
  df_300d: 2,384,617 papers
  Clusters (500d): 50
  Clusters (300d): 50


In [ ]:
# ============================================================
# EXPORT: papers_over_time.json (with corrected years)
# ============================================================

import json

# Group by corrected year and count
papers_by_year = df.groupby('year').size().reset_index(name='count')
papers_by_year = papers_by_year.sort_values('year')

# Filter to reasonable range (2007-2024)
papers_by_year = papers_by_year[
    (papers_by_year['year'] >= 2007) & 
    (papers_by_year['year'] <= 2024)
]

# Convert to list of dicts matching your structure
papers_over_time_data = papers_by_year.to_dict('records')

# Ensure integer types for JSON
papers_over_time_data = [
    {'year': int(row['year']), 'count': int(row['count'])}
    for row in papers_over_time_data
]

# Save to JSON
output_path = 'data/papers_over_time.json'  # Adjust path as needed
with open(output_path, 'w') as f:
    json.dump(papers_over_time_data, f, indent=2)

print(f"✓ Created papers_over_time.json")
print(f"  Year range: {papers_by_year['year'].min()} - {papers_by_year['year'].max()}")
print(f"  Total papers: {papers_by_year['count'].sum():,}")
print(f"\nFirst few years:")
for item in papers_over_time_data[:5]:
    print(f"  {item['year']}: {item['count']:,} papers")

In [19]:
# ============================================================
# EXPORT: clustersizes.json (FIX - Load corrected years)
# ============================================================

import json
import os
import pickle

print("=" * 60)
print("LOADING CORRECTED YEARS AND MERGING INTO df_500d")
print("=" * 60)

# STEP 1: Load the main dataframe with corrected years
# (adjust the path to wherever you saved it)
with open('data/processed/arxiv_metadata_features.pkl', 'rb') as f:
    df_corrected = pickle.load(f)

print(f"Loaded df_corrected: {df_corrected.shape}")
print(f"Columns: {df_corrected.columns.tolist()}")

# STEP 2: Check if years are already corrected in df_corrected
if 'year' not in df_corrected.columns:
    print("Applying year correction to loaded df...")
    
    import re
    
    def extract_original_year(versions_str):
        try:
            if isinstance(versions_str, list) and len(versions_str) > 0:
                first_version = versions_str[0]
                created_date = first_version.get('created', '')
                
                if created_date:
                    year_match = re.search(r'\b(19|20)\d{2}\b', created_date)
                    if year_match:
                        return int(year_match.group())
            return None
        except:
            return None
    
    df_corrected['year'] = df_corrected['versions'].apply(extract_original_year)

print(f"\nYear range in df_corrected: {df_corrected['year'].min()} - {df_corrected['year'].max()}")
print(f"Sample 2014-2016:")
print(df_corrected.groupby('year').size().loc[2014:2016])

# STEP 3: Create mapping from ID to corrected year
print("\nCreating year mapping...")
year_map = df_corrected.set_index('id')['year'].to_dict()
print(f"Year map created with {len(year_map):,} entries")

# STEP 4: Apply corrected years to df_500d
print("\nApplying corrected years to df_500d...")
df_500d['year'] = df_500d['id'].map(year_map)

# Verify
print(f"\n✓ Years updated in df_500d")
print(f"  Year range: {df_500d['year'].min()} - {df_500d['year'].max()}")
print(f"  Missing years: {df_500d['year'].isna().sum()}")
print(f"\n  2014-2016 distribution in df_500d:")
print(df_500d.groupby('year').size().loc[2014:2016])


LOADING CORRECTED YEARS AND MERGING INTO df_500d
Loaded df_corrected: (2384622, 15)
Columns: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed', 'year']

Year range in df_corrected: 2007 - 2025
Sample 2014-2016:
year
2014     97590
2015    105130
2016    113440
dtype: int64

Creating year mapping...
Year map created with 2,384,622 entries

Applying corrected years to df_500d...

✓ Years updated in df_500d
  Year range: 2007 - 2025
  Missing years: 0

  2014-2016 distribution in df_500d:
year
2014     97590
2015    105130
2016    113439
dtype: int64


In [20]:
print("EXPORTING clustersizes.json")
print("=" * 60)

cluster_sizes_data = []
n_clusters = profiles_500d['n_clusters']

for cid in range(n_clusters):
    quality = profiles_500d['quality'][cid]
    top_terms = profiles_500d['top_terms'][cid]
    top_cats = profiles_500d['top_categories'][cid]
    
    # Get meaningful terms for name
    meaningful = [t[0] for t in top_terms[:10] if is_meaningful_term(t[0])][:2]
    if len(meaningful) >= 2:
        display_name = f"{meaningful[0].capitalize()} & {meaningful[1].capitalize()}"
    elif len(meaningful) == 1:
        display_name = meaningful[0].capitalize()
    else:
        display_name = f"{top_terms[0][0].capitalize()} & {top_terms[1][0].capitalize()}"
    
    # Get cluster papers (now with corrected years!)
    cluster_papers = df_500d[df_500d['cluster_id'] == cid].copy()
    
    # ===== TOP CATEGORIES with percentages =====
    cat_dist = top_cats.get('primary_categories', {})
    total_cat = sum(cat_dist.values()) if cat_dist else 1
    top_categories = [
        {'code': cat, 'pct': round(count / total_cat * 100, 1)}
        for cat, count in sorted(cat_dist.items(), key=lambda x: -x[1])[:5]
    ]
    top_category = top_categories[0]['code'] if top_categories else 'Unknown'
    
    # ===== TOP DOMAINS with percentages =====
    # FIX: use category prefix mapping (top_level_domain used raw ArXiv prefixes,
    # causing all physics/math clusters to incorrectly show 'cs' as primary domain)
    cat_dist_c = top_cats.get('primary_categories', {})
    consol_c = {}
    for cc, cnt in cat_dist_c.items():
        d = get_consolidated_domain(cc)
        consol_c[d] = consol_c.get(d, 0) + cnt
    total_domain = sum(consol_c.values()) if consol_c else 1
    top_domains = [
        {
            'code': d,
            'name': DOMAIN_NAMES.get(d, d),
            'count': count,
            'percentage': round(count / total_domain * 100, 1)
        }
        for d, count in sorted(consol_c.items(), key=lambda x: -x[1])
    ]
    primary_domain_code = top_domains[0]['code'] if top_domains else 'other'
    primary_domain = {'code': primary_domain_code, 'name': DOMAIN_NAMES.get(primary_domain_code, primary_domain_code)}
    
    # ===== TOP TERMS with TF-IDF scores =====
    top_terms_with_scores = [
        {'term': t[0], 'score': round(float(t[1]), 4)}
        for t in top_terms[:10]
    ]
    
    # ===== TEMPORAL DATA - papers per year (NOW CORRECTED) =====
    if len(cluster_papers) > 0 and 'year' in cluster_papers.columns:
        valid_years = cluster_papers[
            (cluster_papers['year'] >= 2007) & 
            (cluster_papers['year'] <= 2024) &
            (cluster_papers['year'].notna())
        ]
        
        year_counts = valid_years['year'].value_counts().sort_index()
        
        papers_by_year = [
            {'year': int(year), 'count': int(count)}
            for year, count in year_counts.items()
        ]
    else:
        papers_by_year = []
    
    # ===== MEDIAN YEAR =====
    if len(cluster_papers) > 0 and 'year' in cluster_papers.columns:
        valid_years_median = cluster_papers[
            (cluster_papers['year'] >= 2007) & 
            (cluster_papers['year'] <= 2024) &
            (cluster_papers['year'].notna())
        ]['year']
        median_year = int(valid_years_median.median()) if len(valid_years_median) > 0 else 2020
    else:
        median_year = 2020
    
    # ===== RECENT RATIO =====
    if len(cluster_papers) > 0 and 'year' in cluster_papers.columns:
        valid_years_recent = cluster_papers[
            (cluster_papers['year'] >= 2007) & 
            (cluster_papers['year'] <= 2024) &
            (cluster_papers['year'].notna())
        ]
        if len(valid_years_recent) > 0:
            recent_count = (valid_years_recent['year'] >= 2020).sum()
            recent_ratio = round(recent_count / len(valid_years_recent) * 100, 1)
        else:
            recent_ratio = 0
    else:
        recent_ratio = 0
    
    # ===== GROWTH RATE =====
    if len(papers_by_year) > 0:
        pre_2020 = [p['count'] for p in papers_by_year if p['year'] < 2020]
        post_2020 = [p['count'] for p in papers_by_year if p['year'] >= 2020]
        
        if pre_2020 and post_2020:
            avg_pre = sum(pre_2020) / len(pre_2020)
            avg_post = sum(post_2020) / len(post_2020)
            growth_rate = round((avg_post - avg_pre) / avg_pre * 100, 1) if avg_pre > 0 else 0
        else:
            growth_rate = quality['growth_rate'] * 100
    else:
        growth_rate = quality['growth_rate'] * 100
    
    cluster_sizes_data.append({
        'id': int(cid),
        'name': display_name,
        'size': int(quality['size']),
        'domain': primary_domain,
        'topCategory': top_category,
        'purity': round(quality['category_purity'] * 100, 1),
        'growthRate': round(growth_rate, 1),
        'topTerms': top_terms_with_scores,
        'topCategories': top_categories,
        'topDomains': top_domains,
        'papersByYear': papers_by_year,
        'medianYear': median_year,
        'recentRatio': recent_ratio,
    })

# Sort and save
cluster_sizes_data.sort(key=lambda x: x['id'])

output_dir = 'data'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'clustersizes.json')

with open(output_path, 'w') as f:
    json.dump(cluster_sizes_data, f, indent=2)

print(f"\n✓ Saved: {output_path}")
print(f"  Total clusters: {len(cluster_sizes_data)}")
print(f"\n  VERIFICATION - First 3 clusters:")
for c in cluster_sizes_data[:3]:
    print(f"\n  C{c['id']}: {c['name']}")
    print(f"    Size: {c['size']:,} papers")
    
    # Check 2014-2016
    y2014 = [p for p in c['papersByYear'] if p['year'] == 2014]
    y2015 = [p for p in c['papersByYear'] if p['year'] == 2015]
    y2016 = [p for p in c['papersByYear'] if p['year'] == 2016]
    
    if y2014 and y2015 and y2016:
        ratio_15_14 = y2015[0]['count'] / y2014[0]['count'] if y2014[0]['count'] > 0 else 0
        print(f"    2014: {y2014[0]['count']}, 2015: {y2015[0]['count']}, 2016: {y2016[0]['count']}")
        print(f"    2015/2014 ratio: {ratio_15_14:.2f}x", end='')
        if ratio_15_14 > 1.5:
            print(" ⚠️  SPIKE DETECTED")
        else:
            print(" ✓ Smooth")

EXPORTING clustersizes.json

✓ Saved: data/clustersizes.json
  Total clusters: 50

  VERIFICATION - First 3 clusters:

  C0: Sequence & Number
    Size: 15,947 papers
    2014: 728, 2015: 790, 2016: 834
    2015/2014 ratio: 1.09x ✓ Smooth

  C1: Agent & Learn
    Size: 19,706 papers
    2014: 408, 2015: 384, 2016: 514
    2015/2014 ratio: 0.94x ✓ Smooth

  C2: Data & Model
    Size: 55,038 papers
    2014: 1496, 2015: 1674, 2016: 2048
    2015/2014 ratio: 1.12x ✓ Smooth


In [ ]:
# export evaluation metrics

# eval_metrics is a list of dicts or DataFrame with columns:
# k, silhouette, davies_bouldin, calinski_harabasz, inertia

# Convert to list if it's a DataFrame
if isinstance(eval_metrics_500d, pd.DataFrame):
    metrics_500d_list = eval_metrics_500d.to_dict('records')
else:
    metrics_500d_list = eval_metrics_500d

if isinstance(eval_metrics_300d, pd.DataFrame):
    metrics_300d_list = eval_metrics_300d.to_dict('records')
else:
    metrics_300d_list = eval_metrics_300d

# Extract k values
k_values = [int(m['k']) for m in metrics_500d_list]

# Build elbow data combining both dimensions
elbow_data = []
for i, k in enumerate(k_values):
    m500 = metrics_500d_list[i]
    m300 = metrics_300d_list[i] if i < len(metrics_300d_list) else {}
    
    point = {
        'k': k,
        'inertia500': to_native(m500.get('inertia')),
        'inertia300': to_native(m300.get('inertia')),
        'silhouette500': to_native(m500.get('silhouette')),
        'silhouette300': to_native(m300.get('silhouette')),
        'daviesBouldin500': to_native(m500.get('davies_bouldin')),
        'daviesBouldin300': to_native(m300.get('davies_bouldin')),
        'calinskiHarabasz500': to_native(m500.get('calinski_harabasz')),
        'calinskiHarabasz300': to_native(m300.get('calinski_harabasz')),
    }
    elbow_data.append(point)

chosen_k = 50
chosen_idx = k_values.index(chosen_k) if chosen_k in k_values else -1

export_data = {
    'elbowData': elbow_data,
    'kValues': k_values,
    'chosenK': chosen_k,
    'chosenMetrics': {
        'k': chosen_k,
        'inertia500': to_native(metrics_500d_list[chosen_idx]['inertia']) if chosen_idx >= 0 else None,
        'inertia300': to_native(metrics_300d_list[chosen_idx]['inertia']) if chosen_idx >= 0 else None,
        'silhouette500': to_native(metrics_500d_list[chosen_idx]['silhouette']) if chosen_idx >= 0 else None,
        'silhouette300': to_native(metrics_300d_list[chosen_idx]['silhouette']) if chosen_idx >= 0 else None,
        'daviesBouldin500': to_native(metrics_500d_list[chosen_idx]['davies_bouldin']) if chosen_idx >= 0 else None,
        'calinskiHarabasz500': to_native(metrics_500d_list[chosen_idx]['calinski_harabasz']) if chosen_idx >= 0 else None,
    },
    'methodology': {
        'algorithm': 'K-Means',
        'preprocessing': 'L2 normalization before clustering',
        'note': 'L2 normalization prevents mega-cluster formation in high dimensions. Used standard K-Means (not MiniBatch) for final clustering.',
        'dimensions': [300, 500],
        'metricsUsed': ['silhouette', 'davies_bouldin', 'calinski_harabasz', 'inertia'],
    }
}

with open(os.path.join(output_dir, 'evaluation_metrics.json'), 'w') as f:
    json.dump(export_data, f, indent=2)

print(f"✓ Saved: evaluation_metrics.json")
print(f"  K values tested: {k_values}")
print(f"  Chosen k: {chosen_k}")
if chosen_idx >= 0:
    print(f"  Silhouette (500d, k=50): {metrics_500d_list[chosen_idx]['silhouette']:.4f}")
    print(f"  Davies-Bouldin (500d, k=50): {metrics_500d_list[chosen_idx]['davies_bouldin']:.4f}")


EXPORTING: evaluation_metrics.json
✓ Saved: evaluation_metrics.json
  K values tested: [10, 15, 20, 25, 30, 35, 40, 45, 50]
  Chosen k: 50
  Silhouette (500d, k=50): 0.0277
  Davies-Bouldin (500d, k=50): 4.0589


In [10]:
# export cluster examples

# First, build data for ALL clusters so we can select highlights
all_clusters_data = []

for cid in range(profiles_500d['n_clusters']):
    quality = profiles_500d['quality'][cid]
    temporal = profiles_500d['temporal'][cid]
    top_terms = profiles_500d['top_terms'][cid]
    top_cats = profiles_500d['top_categories'][cid]
    
    # Get meaningful terms for name
    meaningful = [t[0] for t in top_terms[:10] if is_meaningful_term(t[0])][:2]
    if len(meaningful) >= 2:
        display_name = f"{meaningful[0].capitalize()} & {meaningful[1].capitalize()}"
    elif len(meaningful) == 1:
        display_name = meaningful[0].capitalize()
    else:
        display_name = f"{top_terms[0][0].capitalize()} & {top_terms[1][0].capitalize()}"
    
    # Get top category
    cat_dist = top_cats.get('primary_categories', {})
    top_category = max(cat_dist, key=cat_dist.get) if cat_dist else 'Unknown'
    
    # Get primary domain — derived from category prefix (not top_level_domain column)
    cat_dist_raw = profiles_500d['top_categories'][cid].get('primary_categories', {})
    consolidated = {}
    for cc, cnt in cat_dist_raw.items():
        d = get_consolidated_domain(cc)
        consolidated[d] = consolidated.get(d, 0) + cnt
    primary_domain_code = max(consolidated, key=consolidated.get) if consolidated else 'other'
    primary_domain = DOMAIN_NAMES.get(primary_domain_code, primary_domain_code)
    
    # Calculate recent papers percentage
    recent_ratio = temporal['recent_ratio'] * 100  # Already calculated in profiles
    
    # Is it a bridge cluster? (high domain diversity)
    is_bridge = bool(quality['domain_diversity'] > 1.5)
    
    cluster_data = {
        'id': int(cid),
        'name': display_name,
        'topTerms': [t[0] for t in top_terms[:8]],
        'size': to_native(quality['size']),
        'domain': primary_domain,
        'topCategory': top_category,
        'purity': to_native(round(quality['category_purity'] * 100, 1)),
        'growthRate': to_native(round(quality['growth_rate'] * 100, 1)),
        'medianYear': to_native(temporal['median_year']),
        'recentPapers': to_native(round(recent_ratio, 1)),
        'isBridge': is_bridge,
        'domainDiversity': to_native(round(quality['domain_diversity'], 2)),
        'insight': ''  # Will fill in for highlights
    }
    
    all_clusters_data.append(cluster_data)

# --- Select highlights based on interesting characteristics ---

# Sort by different metrics to find interesting clusters
by_growth = sorted(all_clusters_data, key=lambda x: x['growthRate'], reverse=True)
by_size = sorted(all_clusters_data, key=lambda x: x['size'], reverse=True)
by_recency = sorted(all_clusters_data, key=lambda x: x['medianYear'], reverse=True)
by_purity = sorted(all_clusters_data, key=lambda x: x['purity'], reverse=True)
by_diversity = sorted(all_clusters_data, key=lambda x: x['domainDiversity'], reverse=True)
by_slow_growth = sorted(all_clusters_data, key=lambda x: x['growthRate'])

# Pick highlights (avoiding duplicates)
highlight_ids = set()
highlights = []

def add_highlight(cluster, insight):
    if cluster['id'] not in highlight_ids:
        highlight_ids.add(cluster['id'])
        cluster_copy = cluster.copy()
        cluster_copy['insight'] = insight
        highlights.append(cluster_copy)

# Fastest growing (top 2)
for c in by_growth[:2]:
    add_highlight(c, f"Fastest growing cluster — {c['growthRate']:.0f}% growth rate. {c['recentPapers']:.0f}% of papers from 2020+.")

# Largest clusters (top 2)
for c in by_size[:2]:
    if c['id'] not in highlight_ids:
        add_highlight(c, f"One of the largest clusters with {c['size']:,} papers spanning {c['domain']}.")

# Most recent (median year 2023+)
for c in by_recency[:2]:
    if c['id'] not in highlight_ids and c['medianYear'] >= 2022:
        add_highlight(c, f"Very recent cluster — median publication year {c['medianYear']}. {c['recentPapers']:.0f}% from 2020+.")

# Highest purity (well-defined field)
for c in by_purity[:2]:
    if c['id'] not in highlight_ids and c['purity'] > 50:
        add_highlight(c, f"Highly focused cluster — {c['purity']:.0f}% purity in {c['topCategory']}.")

# Most interdisciplinary (bridge clusters)
for c in by_diversity[:2]:
    if c['id'] not in highlight_ids and c['isBridge']:
        add_highlight(c, f"Interdisciplinary bridge — spans multiple domains with diversity score {c['domainDiversity']:.2f}.")

# Slowest growing (mature/declining)
for c in by_slow_growth[:2]:
    if c['id'] not in highlight_ids:
        add_highlight(c, f"Mature field — only {c['growthRate']:.0f}% growth. Established area with stable vocabulary.")

# Limit to ~8-10 highlights
highlights = highlights[:10]

# Remove domainDiversity from final output (was just for selection)
for h in highlights:
    del h['domainDiversity']

with open(os.path.join(output_dir, 'cluster_examples.json'), 'w') as f:
    json.dump(highlights, f, indent=2)

print(f"✓ Saved: cluster_examples.json")
print(f"  Highlights selected: {len(highlights)}")
print(f"\n  Featured clusters:")
for h in highlights:
    print(f"    C{h['id']}: {h['name']}")
    print(f"         {h['insight'][:70]}...")

✓ Saved: cluster_examples.json
  Highlights selected: 10

  Featured clusters:
    C28: Llm & Language
         Fastest growing cluster — 1230350% growth rate. 100% of papers from 20...
    C43: Language & Model
         Fastest growing cluster — 283% growth rate. 86% of papers from 2020+....
    C5: Time & Method
         One of the largest clusters with 234,031 papers spanning Computer Scie...
    C11: Space & Prove
         One of the largest clusters with 180,419 papers spanning Computer Scie...
    C13: Method & Feature
         Very recent cluster — median publication year 2023.0. 83% from 2020+....
    C21: Robot & Human
         Highly focused cluster — 83% purity in cs.RO....
    C39: Planet & Star
         Highly focused cluster — 82% purity in astro-ph.EP....
    C17: Model & Data
         Interdisciplinary bridge — spans multiple domains with diversity score...
    C26: Ray & Gamma ray
         Mature field — only 14% growth. Established area with stable vocabular...
    C2

In [10]:
# export cluster exploration

def get_exploration_badges(cluster_data):
    badges = []
    
    meaningful_count = sum(1 for t in cluster_data.get('topTerms', [])[:10] 
                          if is_meaningful_term(t['term'] if isinstance(t, dict) else t))
    if meaningful_count < 5:
        badges.append({'type': 'warning', 'label': 'Low-Quality Terms', 'description': 'Few meaningful top terms'})
    
    growth = cluster_data.get('growthRate', 0)
    if growth > 100:
        badges.append({'type': 'hot', 'label': 'Explosive Growth', 'description': f'+{growth:.0f}% growth'})
    elif growth > 50:
        badges.append({'type': 'rising', 'label': 'Fast Growing', 'description': f'+{growth:.0f}% growth'})
    elif growth < 20:
        badges.append({'type': 'stable', 'label': 'Mature Field', 'description': 'Slower, steady growth'})
    
    if cluster_data.get('medianYear', 0) >= 2021:
        badges.append({'type': 'new', 'label': 'Very Recent', 'description': 'Most papers from 2021+'})
    
    if cluster_data.get('domainDiversity', 0) > 1.5:
        badges.append({'type': 'bridge', 'label': 'Interdisciplinary', 'description': 'Spans multiple domains'})
    
    size = cluster_data.get('size', 0)
    if size > 100000:
        badges.append({'type': 'large', 'label': 'Major Field', 'description': f'{size:,} papers'})
    elif size < 15000:
        badges.append({'type': 'niche', 'label': 'Niche Area', 'description': f'{size:,} papers'})
    
    return badges

clusters = []

for cid in range(profiles_500d['n_clusters']):
    quality = profiles_500d['quality'][cid]
    temporal = profiles_500d['temporal'][cid]
    top_terms = profiles_500d['top_terms'][cid]
    top_cats = profiles_500d['top_categories'][cid]
    
    terms_with_scores = [
        {'term': t[0], 'score': to_native(t[1]), 'meaningful': is_meaningful_term(t[0])} 
        for t in top_terms[:20]
    ]
    
    meaningful_terms = [t['term'] for t in terms_with_scores if t['meaningful']][:5]
    
    if len(meaningful_terms) >= 2:
        display_name = f"{meaningful_terms[0].capitalize()} & {meaningful_terms[1].capitalize()}"
    elif len(meaningful_terms) >= 1:
        display_name = meaningful_terms[0].capitalize()
    else:
        display_name = f"Cluster {cid}"
    
    cat_dist = top_cats.get('primary_categories', {})
    top_categories = []
    for cat, count in sorted(cat_dist.items(), key=lambda x: x[1], reverse=True)[:10]:
        pct = count / quality['size'] * 100
        top_categories.append({
            'code': cat,
            'name': get_category_name(cat),
            'count': to_native(count),
            'percentage': to_native(round(pct, 2))
        })
    
    # domain distribution — derived from category prefix (not top_level_domain column)
    cat_dist_raw = top_cats.get('primary_categories', {})
    consolidated = {}
    for cc, cnt in cat_dist_raw.items():
        d = get_consolidated_domain(cc)
        consolidated[d] = consolidated.get(d, 0) + cnt
    domain_dist = []
    for domain, count in sorted(consolidated.items(), key=lambda x: -x[1]):
        pct = count / quality['size'] * 100
        domain_dist.append({
            'code': domain,
            'name': get_domain_name(domain),
            'count': to_native(count),
            'percentage': to_native(round(pct, 2))
        })
    
    cluster_data = {
        'id': int(cid),
        'name': display_name,
        'size': to_native(quality['size']),
        'sizePercentage': to_native(round(quality['size'] / len(df_500d) * 100, 2)),
        'topTerms': terms_with_scores,
        'displayTerms': meaningful_terms,
        'hasMeaningfulTerms': len(meaningful_terms) >= 3,
        'topCategories': top_categories,
        'primaryCategory': top_categories[0] if top_categories else None,
        'categoryPurity': to_native(round(top_categories[0]['percentage'], 1)) if top_categories else 0,
        'domainDistribution': domain_dist,
        'primaryDomain': domain_dist[0] if domain_dist else None,
        'domainDiversity': to_native(round(float(quality['domain_diversity']), 3)),
        'yearRange': [to_native(temporal['earliest_year']), to_native(temporal['latest_year'])],
        'medianYear': to_native(temporal['median_year']),
        'recentRatio': to_native(round(float(temporal['recent_ratio']) * 100, 1)),
        'growthRate': to_native(round(float(quality['growth_rate']) * 100, 1)),
        'multiCategoryRate': to_native(round(float(quality['multi_category_rate']) * 100, 1)),
        'distinctiveness': to_native(round(float(quality['distinctiveness']), 3)),
    }
    
    cluster_data['badges'] = get_exploration_badges(cluster_data)
    clusters.append(cluster_data)

clusters.sort(key=lambda x: x['size'], reverse=True)

summary = {
    'totalClusters': len(clusters),
    'totalPapers': to_native(len(df_500d)),
    'avgClusterSize': to_native(len(df_500d) // len(clusters)),
    'largestCluster': max(c['size'] for c in clusters),
    'smallestCluster': min(c['size'] for c in clusters),
    'avgGrowthRate': round(sum(c['growthRate'] for c in clusters) / len(clusters), 1),
    'highGrowthCount': sum(1 for c in clusters if c['growthRate'] > 50),
    'interdisciplinaryCount': sum(1 for c in clusters if c['domainDiversity'] > 1.0),
    'lowQualityTermsCount': sum(1 for c in clusters if not c['hasMeaningfulTerms']),
}

export_data = {
    'clusters': clusters,
    'summary': summary,
}

with open(os.path.join(output_dir, 'cluster_exploration.json'), 'w') as f:
    json.dump(export_data, f, indent=2)

print(f"✓ Saved: cluster_exploration.json")
print(f"  Total clusters: {len(clusters)}")
print(f"  High growth: {summary['highGrowthCount']}")
print(f"  Interdisciplinary: {summary['interdisciplinaryCount']}")
print(f"  Low-quality terms: {summary['lowQualityTermsCount']}")

✓ Saved: cluster_exploration.json
  Total clusters: 50
  High growth: 35
  Interdisciplinary: 31
  Low-quality terms: 0


In [11]:
# export data overview

# Use original dataframe for overview stats
df = df_original

# Papers over time
papers_by_year = df.groupby('year').size()
papers_over_time = [
    {'year': to_native(year), 'count': to_native(count)} 
    for year, count in papers_by_year.items()
    if year >= 1991 and year <= 2024
]

# Category distribution (top 20)
if 'primary_category' in df.columns:
    cat_counts = df['primary_category'].value_counts().head(20)
    category_dist = [
        {
            'category': cat, 
            'name': get_category_name(cat),
            'count': to_native(count), 
            'percentage': to_native(round(count/len(df)*100, 2))
        }
        for cat, count in cat_counts.items()
    ]
else:
    category_dist = []

# Domain distribution — consolidated from primary_category prefix
# (top_level_domain column uses raw ArXiv prefixes like gr-qc, astro-ph, hep-ph
#  which fragment physics into many sub-buckets; consolidate for overview chart)
if 'primary_category' in df.columns:
    consol_overview = {}
    for cat in df['primary_category'].dropna():
        d = get_consolidated_domain(cat)
        consol_overview[d] = consol_overview.get(d, 0) + 1
    domain_dist = [
        {
            'domain': d,
            'name': get_domain_name(d),
            'count': to_native(count),
            'percentage': to_native(round(count / len(df) * 100, 2))
        }
        for d, count in sorted(consol_overview.items(), key=lambda x: -x[1])
    ]
elif 'top_level_domain' in df.columns:
    # fallback: use top_level_domain if primary_category absent
    domain_counts = df['top_level_domain'].value_counts()
    domain_dist = [
        {'domain': d, 'name': get_domain_name(d), 'count': to_native(c),
         'percentage': to_native(round(c/len(df)*100, 2))}
        for d, c in domain_counts.items()
    ]
else:
    domain_dist = []

export_data = {
    'papersOverTime': papers_over_time,
    'categoryDistribution': category_dist,
    'domainDistribution': domain_dist,
    'summary': {
        'totalPapers': to_native(len(df)),
        'yearRange': [to_native(df['year'].min()), to_native(df['year'].max())],
        'uniqueCategories': to_native(df['primary_category'].nunique()) if 'primary_category' in df.columns else 0,
        'uniqueDomains': len(consol_overview) if 'primary_category' in df.columns else (to_native(df['top_level_domain'].nunique()) if 'top_level_domain' in df.columns else 0),
    }
}

with open(os.path.join(output_dir, 'data_overview.json'), 'w') as f:
    json.dump(export_data, f, indent=2)

print(f"✓ Saved: data_overview.json")
print(f"  Total papers: {export_data['summary']['totalPapers']:,}")
print(f"  Year range: {export_data['summary']['yearRange']}")
print(f"  Unique categories: {export_data['summary']['uniqueCategories']}")


✓ Saved: data_overview.json
  Total papers: 2,384,617
  Year range: [2007, 2025]
  Unique categories: 153


In [15]:
import joblib

with open('data/processed/svd_variance_info_500.pkl', 'rb') as f:
    variance_500d = joblib.load(f)

# The pkl likely contains explained_variance_ratio_ array from SVD
# We need to calculate cumulative variance at specific component counts

# If it's an array of variance ratios per component:
if isinstance(variance_500d, np.ndarray):
    variance_ratios = variance_500d
elif isinstance(variance_500d, dict) and 'explained_variance_ratio' in variance_500d:
    variance_ratios = variance_500d['explained_variance_ratio']
elif hasattr(variance_500d, 'explained_variance_ratio_'):
    variance_ratios = variance_500d.explained_variance_ratio_
else:
    # Try to use it directly
    variance_ratios = np.array(variance_500d)

# Calculate cumulative variance
cumulative = np.cumsum(variance_ratios) * 100  # Convert to percentage

# Sample at specific component counts
component_samples = [1, 5, 10, 25, 50, 100, 200, 300, 400, 500]

variance_data = []
for n in component_samples:
    if n <= len(cumulative):
        variance_data.append({
            'components': n,
            'cumulative_variance': round(float(cumulative[n-1]), 2)
        })

with open(os.path.join(output_dir, 'cumulative_variance.json'), 'w') as f:
    json.dump(variance_data, f, indent=2)

print(f"✓ Saved: cumulative_variance.json")
print(f"  Sample points: {len(variance_data)}")
for v in variance_data:
    print(f"    {v['components']} components: {v['cumulative_variance']}%")

✓ Saved: cumulative_variance.json
  Sample points: 10
    1 components: 0.33%
    5 components: 4.05%
    10 components: 6.74%
    25 components: 12.53%
    50 components: 19.36%
    100 components: 29.52%
    200 components: 44.47%
    300 components: 56.0%
    400 components: 65.48%
    500 components: 73.45%


In [24]:
# ============================================================
# ADD DESCRIPTIONS TO CLUSTERSIZES.JSON
# ============================================================

import json

# All 50 cluster descriptions
descriptions = [
    {"id": 0, "description": "Research spanning number theory, combinatorics, and sequence analysis, with particular focus on properties of integer sequences and their mathematical structures. The term 'sequence' appears as the dominant feature, reflecting work on mathematical patterns, counting problems, and algorithmic enumeration. Showing moderate growth (61%), this represents a stable mathematical core with increasing computational applications in machine learning and data structures."},
    {"id": 1, "description": "The epicenter of reinforcement learning and multi-agent systems, studying how autonomous agents learn optimal behaviors through interaction with environments. With explosive 152% growth, this cluster captures the surge in game-playing AI, multi-agent coordination, and policy optimization methods that have revolutionized robotics and strategic decision-making since 2015."},
    {"id": 2, "description": "Broad methodological research on data analysis, statistical modeling, and machine learning pipelines—the 'how-to' papers of modern data science. The low purity (16.2%) reflects its role as a bridge between statistics, computer science, and domain applications. Growing steadily at 98%, this cluster represents the practical toolkit used across scientific computing."},
    {"id": 3, "description": "Quantum information science and quantum computing, dominated by research on quantum states, entanglement, and circuit design. With remarkably high purity (65.0%) and strong growth (119%), this represents one of the most cohesive research communities in modern physics, driven by advances in quantum hardware and the race toward quantum advantage."},
    {"id": 4, "description": "Neutrino physics and particle detection, studying the properties of these elusive particles through oscillation experiments and mass measurements. The exceptionally high purity (53.6%) reflects a tightly-knit experimental physics community, while modest growth (29%) indicates a mature field with steady, methodical progress rather than explosive breakthroughs."},
    {"id": 5, "description": "Stochastic processes, dynamical systems, and time-series analysis—mathematical methods for studying systems that evolve randomly or deterministically over time. The low purity (5.5%) reveals this as highly interdisciplinary, with applications from statistical mechanics to financial modeling. Moderate 61% growth reflects its role as essential mathematical infrastructure across sciences."},
    {"id": 6, "description": "Extragalactic astronomy and galaxy evolution, examining how galaxies form, grow, and cluster across cosmic time using observations at multiple wavelengths. The strong focus on redshift measurements and stellar masses reflects observational cosmology's empirical foundation. Moderate 54% growth tracks the steady expansion of survey data from next-generation telescopes."},
    {"id": 7, "description": "Core machine learning research on model architectures, training algorithms, and deep learning theory—the methodological backbone of modern AI. With exceptional 96% growth and recent median year (2022), this captures the deep learning revolution's second wave, moving beyond initial breakthroughs to systematic understanding of why and how neural networks work."},
    {"id": 8, "description": "Spintronics and magnetic phenomena in condensed matter, studying how electron spins can be manipulated for information storage and processing. Terms like 'spin-orbit coupling' and 'magnetic field' point to both fundamental physics and technological applications in next-generation electronics. Moderate 44% growth reflects steady progress in a technically challenging field."},
    {"id": 9, "description": "Optimization theory and algorithm design, focused on solving complex computational problems efficiently. The prominence of 'convergence' and 'optimal solution' reveals concern with both theoretical guarantees and practical performance. Solid 66% growth driven by machine learning's insatiable demand for better optimizers and the resurgence of convex optimization methods."},
    {"id": 10, "description": "Pure mathematics exploring function spaces, analytic methods, and number-theoretic structures. The diverse mix of 'function,' 'integral,' 'formula,' and 'prove' reflects classical analysis combined with modern computational approaches. Moderate 50% growth suggests steady evolution as computational tools enable attacks on longstanding mathematical problems."},
    {"id": 11, "description": "Core pure mathematics spanning algebraic geometry, number theory, and combinatorics—the theoretical spine of modern mathematics. The prevalence of proof-oriented language ('prove,' 'theorem,' 'conjecture') marks this as foundational research. With low purity (10.9%), these mathematical structures appear across many domains, showing steady 58% growth as computational methods open new avenues."},
    {"id": 12, "description": "Black hole physics and general relativity, studying spacetime singularities, event horizons, and gravitational phenomena in strong-field regimes. The moderate purity (45.9%) reflects black holes' central role connecting classical relativity, quantum mechanics, and astrophysical observations. Growth of 66% accelerated by gravitational wave detections since 2015."},
    {"id": 13, "description": "Computer vision and visual recognition research, dominated by convolutional architectures, object detection, and image segmentation methods. The exceptionally high purity (52.1%) and explosive 160% growth make this one of deep learning's flagship success stories, transforming from academic curiosity to ubiquitous technology in under a decade."},
    {"id": 14, "description": "Human-computer interaction, software engineering, and sociotechnical systems—research on how people design, build, and use computational systems. The low purity (9.0%) reflects genuine interdisciplinarity spanning CS, social science, and design. Strong 103% growth driven by AI ethics concerns, collaborative software development, and usability research for emerging technologies."},
    {"id": 15, "description": "Observational astrophysics focusing on electromagnetic emission across the spectrum, from radio to X-ray wavelengths. The terms 'disk,' 'dust,' and 'radio' point to studies of stellar and galactic environments using multi-wavelength observations. Modest 37% growth reflects the mature observational astronomy community's steady expansion of telescope capabilities."},
    {"id": 16, "description": "Differential geometry and Riemannian manifolds, studying curved spaces through metrics, curvatures, and topological invariants. High purity (37.0%) indicates a cohesive mathematical community, while 56% growth suggests renewed interest driven by applications in general relativity, data science (manifold learning), and theoretical physics."},
    {"id": 17, "description": "Statistical modeling, Bayesian inference, and probabilistic machine learning—research on uncertainty quantification and model parameter estimation. The low purity (10.8%) reflects widespread adoption across disciplines from epidemiology to climate science. Strong 105% growth powered by probabilistic deep learning and the integration of domain knowledge with data-driven methods."},
    {"id": 18, "description": "Condensed matter physics studying phase transitions, critical phenomena, and emergent order in materials. The emphasis on 'temperature,' 'topological,' and 'quantum' reveals both classical thermodynamics and modern topological phases of matter. Moderate 54% growth reflects steady progress in understanding exotic materials and quantum phase transitions."},
    {"id": 19, "description": "Cybersecurity and adversarial machine learning, studying attacks on ML systems and defenses against them. The terms 'adversarial,' 'robustness,' and 'security' highlight concerns about AI safety and reliability. Explosive 114% growth reflects urgent concerns as ML systems move from research to deployment in security-critical applications."},
    {"id": 20, "description": "Partial differential equations and mathematical analysis of boundary value problems, fundamental to modeling physical phenomena. High purity (35.1%) indicates strong mathematical foundations, while moderate 50% growth suggests steady evolution as computational methods enable solving previously intractable nonlinear problems."},
    {"id": 21, "description": "Robotics and embodied AI, studying how robots perceive environments, plan actions, and interact with humans in real-world settings. Exceptionally high purity (82.9%) marks robotics as a distinct research community, while staggering 165% growth reflects the convergence of better sensors, stronger compute, and learning-based control revolutionizing manipulation and navigation."},
    {"id": 22, "description": "Neural network architectures, graph neural networks, and network science—spanning both artificial neural nets and complex networks in nature and society. The low purity (18.0%) reflects this as a bridge between deep learning, social network analysis, and neuroscience. Moderate 43% growth indicates maturation as graph neural networks become standard tools."},
    {"id": 23, "description": "Graph theory and discrete mathematics, studying combinatorial structures through vertices, edges, and connectivity properties. The high purity (35.4%) reflects a cohesive mathematical community, while strong 87% growth is driven by applications in machine learning (graph neural networks), social network analysis, and algorithm design on complex networks."},
    {"id": 24, "description": "Algebraic number theory and polynomial algebra, studying special functions, coefficient properties, and number-theoretic relationships. The relatively low purity (13.4%) suggests these mathematical tools serve diverse applications from coding theory to combinatorics. Moderate 53% growth indicates steady evolution as computational algebra opens new research directions."},
    {"id": 25, "description": "High-energy theoretical physics, studying quantum field theory, gauge theories, and string theory's mathematical foundations. The moderate purity (38.9%) reflects connections between particle physics and pure mathematics. With 46% growth, this mature field continues evolving through new mathematical techniques and connections to condensed matter physics."},
    {"id": 26, "description": "Gamma-ray astronomy and high-energy astrophysics, detecting the universe's most violent phenomena through energetic photons. The high purity (56.9%) reflects specialized observational techniques, while modest 14% growth indicates a mature field with steady expansion of detector capabilities rather than revolutionary breakthroughs."},
    {"id": 27, "description": "Quantum optics and photonics, studying light-matter interactions at the quantum level through lasers, optical cavities, and photonic devices. The moderate purity (24.2%) reflects applications spanning quantum information, telecommunications, and precision measurement. Steady 48% growth driven by quantum technology development and integrated photonics advances."},
    {"id": 28, "description": "The large language model revolution: research on transformer architectures, prompting strategies, reasoning capabilities, and alignment of billion-parameter models. With astronomical 1,230,350% growth (essentially starting from zero pre-2022), this cluster captures AI's most dramatic recent transformation, from GPT-3 through ChatGPT to modern foundation models."},
    {"id": 29, "description": "Particle physics and collider phenomenology, studying quarks, Higgs bosons, and fundamental interactions through high-energy experiments. The high purity (53.8%) reflects the particle physics community's cohesion around major facilities like the LHC. Modest 19% growth indicates a mature experimental program with steady but incremental progress post-Higgs discovery."},
    {"id": 30, "description": "Linear algebra, matrix theory, and numerical methods for large-scale computation—the computational backbone of modern data science. The low purity (8.2%) reflects ubiquitous use across all quantitative fields. Moderate 51% growth driven by demands from machine learning for efficient matrix factorizations, eigenvalue solvers, and randomized algorithms."},
    {"id": 31, "description": "Research using Greek-letter notation common across physics and mathematics—a cluster defined more by symbolic conventions than specific domain. The low purity (9.7%) and diverse category distribution suggest this captures papers with heavy mathematical formalism spanning fields. Moderate 55% growth reflects steady mathematical research output."},
    {"id": 32, "description": "Wave phenomena and gravitational wave physics, studying how disturbances propagate through media or spacetime itself. The terms span classical wave theory, nonlinear dynamics, and relativistic gravity. Moderate 42% growth accelerated by LIGO's detections, opening a new window on the universe through gravitational wave astronomy."},
    {"id": 33, "description": "Representation theory and Lie algebras, studying symmetries through algebraic structures and their matrix representations. The moderate purity (17.2%) reflects deep connections between abstract algebra, geometry, and theoretical physics. Steady 52% growth driven by applications in particle physics, integrable systems, and geometric representation theory."},
    {"id": 34, "description": "Video understanding and temporal modeling, extending computer vision to dynamic scenes through frame sequences and temporal reasoning. Exceptionally high purity (72.6%) indicates a specialized community, while explosive 158% growth reflects the video AI boom driven by self-supervised learning, transformers for video, and multimodal foundation models."},
    {"id": 35, "description": "Stellar physics and stellar populations, studying the formation, evolution, and death of individual stars through spectroscopy and photometry. High purity (43.2%) reflects observational astronomy's star-focused community. Modest 37% growth indicates steady progress characterizing stellar properties and understanding galactic chemical evolution through stellar archaeology."},
    {"id": 36, "description": "Magnetism and magnetic materials, studying ordered magnetic states, magnetic fields' effects on matter, and spintronic phenomena. The low purity (13.4%) reflects magnetic phenomena's ubiquity from condensed matter physics to astrophysics. Moderate 39% growth driven by interest in exotic magnetic materials and magnetic data storage technologies."},
    {"id": 37, "description": "Computational complexity and algorithm analysis, studying asymptotic behavior and efficiency bounds using big-O notation. The moderate purity (15.9%) reflects theoretical computer science's connections to discrete mathematics. Solid 65% growth driven by machine learning's demand for faster algorithms and renewed interest in fine-grained complexity."},
    {"id": 38, "description": "Medical imaging, remote sensing, and applied computer vision where image analysis solves domain-specific problems. The very high purity (55.6%) reflects specialized applications beyond general computer vision. Exceptional 111% growth powered by deep learning's success on medical diagnosis, satellite imagery analysis, and scientific image processing."},
    {"id": 39, "description": "Exoplanet science and planetary systems, studying worlds beyond our solar system through transit detections, radial velocities, and direct imaging. Exceptionally high purity (81.7%) marks this as a distinct observational astronomy subdiscipline. Moderate 30% growth reflects maturation after Kepler's prolific discoveries, now focusing on atmospheric characterization and habitability."},
    {"id": 40, "description": "Materials science and condensed matter, studying electronic structure, phase behavior, and properties of solids from first principles and experiment. The moderate purity (21.1%) spans computational materials design, experimental synthesis, and device applications. Steady 38% growth driven by the search for new functional materials including superconductors, topological insulators, and 2D materials."},
    {"id": 41, "description": "Dark matter and dark energy research, studying the mysterious components comprising 95% of the universe through cosmological observations and particle physics. The moderate purity (36.9%) reflects both observational cosmology and theoretical particle physics approaches. Solid 59% growth driven by improved cosmological measurements and ongoing direct detection experiments."},
    {"id": 42, "description": "Information theory and wireless communications, studying fundamental limits of data transmission through noisy channels. The high purity (44.3%) reflects communication theory's mathematical foundations. Steady 60% growth driven by 5G/6G research, quantum communication channels, and the convergence of information theory with machine learning."},
    {"id": 43, "description": "Natural language processing and computational linguistics before the LLM era, studying language models, pre-training, fine-tuning, and linguistic structure. With stunning 283% growth, this captures the transformer revolution (2017-2022) that preceded the LLM explosion—BERT, GPT-2/3, and the shift to pre-trained models."},
    {"id": 44, "description": "Research employing lambda calculus notation and Greek-letter variables common in formal mathematics and theoretical physics. The low purity (8.9%) and diverse domains suggest this cluster captures papers with heavy mathematical formalism rather than a specific research area. Moderate 53% growth tracks overall mathematical research output."},
    {"id": 45, "description": "Speech recognition, synthesis, and audio processing using deep learning architectures. The moderate purity (33.5%) spans signal processing and NLP applications. Explosive 139% growth driven by end-to-end neural models, attention mechanisms for speech, and speech's integration into multimodal AI systems like voice assistants."},
    {"id": 46, "description": "Group theory and algebraic structures, studying symmetries, group actions, and representations through abstract algebra. The moderate purity (23.0%) reflects group theory's role as mathematical infrastructure across physics, cryptography, and combinatorics. Steady 60% growth indicates continued theoretical development and new applications in quantum computing and cryptography."},
    {"id": 47, "description": "Functional analysis and operator theory, studying infinite-dimensional spaces and linear operators with applications from quantum mechanics to PDEs. The moderate purity (17.7%) reflects these tools' utility across mathematical physics. Steady 57% growth driven by quantum information theory and spectral methods for differential equations."},
    {"id": 48, "description": "Fluid dynamics and turbulence, studying the complex flow of liquids and gases from viscous flows to turbulent chaos. The high purity (27.6%) reflects computational and theoretical fluid mechanics. Solid 66% growth driven by climate modeling, aerodynamic design, and machine learning for turbulence modeling and flow control."},
    {"id": 49, "description": "Research using gamma-ray notation and decay processes, spanning particle physics, nuclear physics, and mathematical analysis with Greek letters. The low purity (17.8%) suggests this captures mathematical physics papers using common symbolic conventions. Moderate 44% growth tracks general physics research with heavy formalism."}
]

# Load current clustersizes.json
with open('results/clustersizes.json', 'r') as f:
    clusters = json.load(f)

print(f"Loaded {len(clusters)} clusters")
print(f"Have {len(descriptions)} descriptions")

# Create lookup dict
desc_lookup = {d['id']: d['description'] for d in descriptions}

# Add descriptions
for cluster in clusters:
    cluster_id = cluster['id']
    if cluster_id in desc_lookup:
        cluster['description'] = desc_lookup[cluster_id]
    else:
        print(f"⚠️  No description for cluster {cluster_id}")
        cluster['description'] = ""

# Save
with open('data/clustersizes.json', 'w') as f:
    json.dump(clusters, f, indent=2)

print(f"\n✓ Updated {len(clusters)} clusters with descriptions")

Loaded 50 clusters
Have 50 descriptions

✓ Updated 50 clusters with descriptions


In [25]:
# Check if descriptions are in the file
import json

with open('data/clustersizes.json', 'r') as f:
    clusters = json.load(f)

print(f"Total clusters: {len(clusters)}")
print(f"\nChecking first 3 clusters for descriptions:")

for i in range(3):
    c = clusters[i]
    print(f"\n--- Cluster {c['id']}: {c['name']} ---")
    print(f"Keys in this cluster: {list(c.keys())}")
    
    if 'description' in c:
        print(f"✓ Has description: {c['description'][:100]}...")
    else:
        print(f"✗ NO DESCRIPTION FIELD!")

# Also check the actual JSON structure
print("\n\nFirst cluster raw structure:")
print(json.dumps(clusters[0], indent=2)[:500])

Total clusters: 50

Checking first 3 clusters for descriptions:

--- Cluster 0: Sequence & Number ---
Keys in this cluster: ['id', 'name', 'size', 'domain', 'topCategory', 'purity', 'growthRate', 'topTerms', 'topCategories', 'topDomains', 'papersByYear', 'medianYear', 'recentRatio', 'description']
✓ Has description: Research spanning number theory, combinatorics, and sequence analysis, with particular focus on prop...

--- Cluster 1: Agent & Learn ---
Keys in this cluster: ['id', 'name', 'size', 'domain', 'topCategory', 'purity', 'growthRate', 'topTerms', 'topCategories', 'topDomains', 'papersByYear', 'medianYear', 'recentRatio', 'description']
✓ Has description: The epicenter of reinforcement learning and multi-agent systems, studying how autonomous agents lear...

--- Cluster 2: Data & Model ---
Keys in this cluster: ['id', 'name', 'size', 'domain', 'topCategory', 'purity', 'growthRate', 'topTerms', 'topCategories', 'topDomains', 'papersByYear', 'medianYear', 'recentRatio', 'descript